# Clase 2 · Laboratorio — Diseño dimensional para Saber 11

**Trabajo en parejas · 90 min.**

Al final deben tener un notebook completado con las 4 tareas resueltas y subirlo a Moodle antes de las 23:59.

**Checkpoint conjunto a los 55 min:** el profesor detiene la sala y revisamos juntos la Tarea 3 (hecho + FKs) antes de pasar al diseño del proyecto propio.

In [1]:
import pandas as pd

df = pd.read_csv("../saber11_muestra_500k.csv")
print(f"Cargados {len(df):,} registros — {df.shape[1]} columnas")

Cargados 500,000 registros — 22 columnas


## Tarea 1 (15 min) — dim_colegio con clave surrogate

Construye una tabla de dimensión `dim_colegio` que contenga:
- Una clave surrogate `colegio_id` (entero secuencial).
- Los atributos: COLE_NATURALEZA, COLE_JORNADA, COLE_CALENDARIO, COLE_BILINGUE.

Requisitos:
- Sin duplicados (una fila por combinación única de los 4 atributos).
- Verifica que la clave surrogate es única.

In [2]:
# construyo dim_colegio con las 4 columnas indicadas y una clave surrogate colegio_id
cols_colegio = ["COLE_NATURALEZA", "COLE_JORNADA", "COLE_CALENDARIO", "COLE_BILINGUE"]

dim_colegio = df[cols_colegio].drop_duplicates().reset_index(drop=True)
dim_colegio["colegio_id"] = dim_colegio.index + 1  # empiezo en 1, no en 0

# Validar unicidad de la surrogate
assert dim_colegio["colegio_id"].is_unique, "La clave surrogate NO es única"
print(f"dim_colegio: {len(dim_colegio)} filas")
dim_colegio.head()

dim_colegio: 40 filas


,COLE_NATURALEZA,COLE_JORNADA,COLE_CALENDARIO,COLE_BILINGUE,colegio_id
0,OFICIAL,COMPLETA,A,N,1
1,NO OFICIAL,NOCHE,A,N,2
2,OFICIAL,TARDE,A,N,3
3,OFICIAL,MAÑANA,A,N,4
4,NO OFICIAL,TARDE,A,N,5


## Tarea 2 (15 min) — dim_geografia con jerarquía

Construye `dim_geografia` con:
- Clave surrogate `geo_id`.
- Atributos: COLE_DEPTO_UBICACION (padre), COLE_MCPIO_UBICACION (hijo).

Requisitos:
- Una fila por par (departamento, municipio).
- Documenta en Markdown por qué la jerarquía es útil para el análisis.

In [3]:
# dim_geografia: una fila por combinacion (departamento, municipio)
cols_geo = ["COLE_DEPTO_UBICACION", "COLE_MCPIO_UBICACION"]

dim_geografia = df[cols_geo].drop_duplicates().reset_index(drop=True)
dim_geografia["geo_id"] = dim_geografia.index + 1

assert dim_geografia["geo_id"].is_unique
print(f"dim_geografia: {len(dim_geografia)} filas")
dim_geografia

dim_geografia: 10 filas


,COLE_DEPTO_UBICACION,COLE_MCPIO_UBICACION,geo_id
0,ATLÁNTICO,BARRANQUILLA,1
1,BOLÍVAR,CARTAGENA,2
2,BOGOTÁ D.C.,BOGOTÁ D.C.,3
3,NARIÑO,PASTO,4
4,CÓRDOBA,MONTERÍA,5
5,ANTIOQUIA,MEDELLÍN,6
6,SANTANDER,BUCARAMANGA,7
7,VALLE DEL CAUCA,CALI,8
8,CUNDINAMARCA,SOACHA,9
9,TOLIMA,IBAGUÉ,10


**¿Por qué sirve la jerarquía depto -> municipio?**

Porque asi despues en el analisis puedo "subir" de nivel (hacer drill-up) y sacar promedios por departamento sin tener que volver a tocar la tabla de hechos, simplemente agrupando por geo_id y trayendo el departamento desde dim_geografia. Tambien sirve para filtrar rapido: si el profesor pregunta por un departamento especifico, ya tengo esa columna lista en la dimension en vez de tenerla repetida 500,000 veces en la tabla de hechos.

Nota curiosa: en este dataset de muestra solo hay 10 departamentos y cada uno tiene un solo municipio (parece que solo usaron la capital de cada deptp), por eso dim_geografia me quedo con exactamente 10 filas. En un dataset real del ICFES completo si habria muchos mas municipios por departamento.

## Tarea 3 (25 min) — hecho_resultados

Construye la tabla de hechos `hecho_resultados` con:
- FKs a: `dim_colegio` (colegio_id), `dim_geografia` (geo_id), `dim_tiempo` (tiempo_id — bosqueja también esta dimensión).
- Medidas: PUNT_LECTURA_CRITICA, PUNT_MATEMATICAS, PUNT_C_NATURALES, PUNT_SOCIALES_CIUDADANAS, PUNT_INGLES, PUNT_GLOBAL.

Requisitos:
- La granularidad del hecho es "una fila por estudiante-periodo".
- Después de hacer los joins con las dimensiones, la tabla de hechos NO debe perder filas (comparar `len(hecho_resultados)` con `len(df)` original).

In [4]:
# 1) dim_tiempo (boceto)
dim_tiempo = df[["PERIODO"]].drop_duplicates().reset_index(drop=True)
dim_tiempo["tiempo_id"] = dim_tiempo.index + 1

# 2) uno df con las 3 dimensiones para sacar las FKs (uso left join para no perder filas)
medidas = [
    "PUNT_LECTURA_CRITICA",
    "PUNT_MATEMATICAS",
    "PUNT_C_NATURALES",
    "PUNT_SOCIALES_CIUDADANAS",
    "PUNT_INGLES",
    "PUNT_GLOBAL",
]

hecho_resultados = (
    df.merge(dim_colegio, on=cols_colegio, how="left")
      .merge(dim_geografia, on=cols_geo, how="left")
      .merge(dim_tiempo, on="PERIODO", how="left")
      [["colegio_id", "geo_id", "tiempo_id"] + medidas]
)

# 3) validar que no perdimos filas
assert len(hecho_resultados) == len(df), f"Perdimos filas: {len(df) - len(hecho_resultados)}"
print(f"hecho_resultados: {len(hecho_resultados)} filas (esperado {len(df)})")
hecho_resultados.head()

hecho_resultados: 500000 filas (esperado 500000)


,colegio_id,geo_id,tiempo_id,PUNT_LECTURA_CRITICA,PUNT_MATEMATICAS,PUNT_C_NATURALES,PUNT_SOCIALES_CIUDADANAS,PUNT_INGLES,PUNT_GLOBAL
0,1,1,1,48,51,26,14,67,194
1,1,2,2,77,32,64,48,48,199
2,2,3,3,65,43,63,84,48,268
3,3,4,4,61,45,52,62,49,365
4,3,2,4,68,87,52,16,57,355


**Nota sobre nulos en las FKs:** en este caso no me dio ningun nulo porque las 3 dimensiones se construyeron directamente a partir de las columnas del mismo `df`, entonces todas las combinaciones que aparecen en el hecho ya existen en las dimensiones. Si estuviera cargando las dimensiones desde otra fuente (por ejemplo un catalogo de colegios aparte), ahi si podria pasar que un colegio del hecho no exista en la dimension y me quede colegio_id en null — en ese caso tocaria revisar si es un dato mal escrito o si falta agregarlo a la dimension.

## ⏸ Checkpoint del profesor (10 min)

Detengan aquí. El profesor revisa la Tarea 3 con toda la sala: cómo evitar perder filas al hacer joins, qué hacer si aparecen nulos en las FKs.

## Tarea 4 (30 min) — Modelo dimensional del proyecto del grupo

Con tu grupo, diseñen el modelo dimensional para su dataset del proyecto:

1. Identifiquen el **hecho** principal y su granularidad.
2. Identifiquen 3-4 **dimensiones** y justifiquen brevemente cada una.
3. Dibujen el modelo (celda Markdown con diagrama tipo Mermaid, ASCII, o adjunten un PNG de draw.io).

Al final, guarden el diagrama en el repo del grupo bajo `entrega/fase_a_diseño_borrador.pdf`.

### Diagrama de nuestro grupo

Ya confirmamos el dataset del proyecto con mi grupo: siniestralidad vial en Cali (2016-2024), el mismo `siniestralidad_2016_2024.csv` que documentamos a fondo en la Fase A (`entrega/fase_a_diseno.pdf`). Acá dejo el resumen de ese diseño.

**1. Hecho principal y granularidad**

`FACT_SINIESTRO` — el grano es una fila = un siniestro vial reportado (identificado por el Código original de la fuente). Elegimos este grano porque es el nivel más atómico que hay en los datos y el que piden casi todas nuestras preguntas de negocio (conteos y tasas de siniestros); si agrupáramos a un grano más alto (por ejemplo uno por día) ya no podríamos analizar despues la gravedad o el tipo de vehículo de un caso puntual.

**2. Dimensiones (5 en total, más una extensión)**

- `DIM_TIEMPO` (grano: un día calendario): para poder ver cómo evoluciona la siniestralidad por año, mes y día de la semana, en vez de operar directo sobre el campo Fecha en texto.
- `DIM_GRAVEDAD`: para estandarizar la columna "Tipo confirmado", que trae variantes como "Solo daños" y "Solo Daños" como si fueran categorías distintas, y para poder ordenar la gravedad (Negativo < Solo Daños < Con Heridos < Con Fallecidos).
- `DIM_TIPO_ACCIDENTE`: agrupa la clase del accidente (choque, atropello, volcamiento...); en la fuente esta info viene duplicada en dos columnas y con texto sucio ("Choque"/"CHOQUE"), asi que conviene limpiarla una sola vez en la dimensión.
- `DIM_UBICACION` (grano: una dirección de reporte única): para responder qué vías/direcciones acumulan más siniestros. La dejamos de un solo nivel porque la fuente solo trae texto libre de dirección (casi la mitad de los datos nulos), no hay como armar una jerarquía tipo ciudad/comuna confiable.
- `DIM_MEDIO_REPORTE`: para ver qué canal de reporte predomina (línea 123, línea 127, radio, agente de tránsito, redes sociales) y cómo se relaciona con la gravedad del caso.

Extra (fuera del modelo estrella puro): la columna "Tipo de vehículos implicados" es multivaluada (844 combinaciones distintas de ~55 tipos de vehículo, ej. "AUTOMOVIL,MOTOCICLETA"), asi que no cabe como columna plana del hecho. Para eso el diseño agrega una extensión de copo de nieve: `DIM_TIPO_VEHICULO` + una tabla puente `BRIDGE_SINIESTRO_VEHICULO` (con una columna de peso para no duplicar conteos). Esta parte no es modelo estrella puro, pero es la única excepción — el resto de la bodega sí lo es.

**3. Diagrama**

```
                              DIM_TIEMPO
                    (id_tiempo, fecha, anio, trimestre,
                     mes, dia_semana, es_fin_de_semana)
                                   |
DIM_GRAVEDAD                      |                      DIM_TIPO_ACCIDENTE
(id_gravedad,                     |                      (id_tipo_accidente,
gravedad_estandarizada, -----  FACT_SINIESTRO  -----     tipo_accidente_estandarizado)
nivel_severidad)                (id_siniestro (PK degenerada),
                                  id_tiempo (FK), id_gravedad (FK),
                                  id_tipo_accidente (FK), id_ubicacion (FK),
                                  id_medio_reporte (FK),
                                  numero_vehiculos, cantidad_siniestros)
                                   |                      |
                            DIM_UBICACION           DIM_MEDIO_REPORTE
                        (id_ubicacion,              (id_medio_reporte,
                         direccion_reporte,          medio_reporte_estandarizado,
                         tipo_via_principal)          es_reporte_agente)

  (extensión copo de nieve, no modelo estrella puro)
  FACT_SINIESTRO --< BRIDGE_SINIESTRO_VEHICULO >-- DIM_TIPO_VEHICULO
                     (id_siniestro FK, id_tipo_vehiculo FK, peso)
```

Guardamos el diseño completo (con la justificación de estrella vs. copo de nieve y el detalle de cada atributo) en `entrega/fase_a_diseno.pdf`.